# Probing and attention

**Track:** Frontier research. **Level:** Advanced. **CPU:** yes.

## Goals

- Train a one-layer attention model whose target at every position is the token in position 0.
- Measure how much attention mass lands on position 0.
- Compare that model with a probe that sees only the current token.
- Run the same tokens through TransformerLens `run_with_cache` when the package is installed, and name the cache tensors that correspond to the measurement.

## Sources and license

Original notebook. Linear probes are Alain and Bengio, *Understanding Intermediate Layers Using Linear Classifier Probes* (2016). The attention measurement is the kind of circuit check in Elhage et al., *A Mathematical Framework for Transformer Circuits* (Anthropic, 2021). The cache API is TransformerLens (Neel Nanda, MIT). This lab does not copy ARENA or TransformerLens tutorials. If `transformer_lens` is not installed the cache cell prints SKIP and the rest of the lab still runs. CPU CI does not require the package, because the measurement above is the experiment. The install line is in the further-reading cell as a comment, not as a hidden download.

## Concepts

A probe is a linear classifier on a frozen activation. It answers "is this fact linearly readable here?" It does not answer "does the model use it?". Attention mass answers a more specific question: of the residual update at this layer, which source positions is the head averaging? On this task the label is the token at position 0, and a causal head is allowed to look at position 0 from every later position. A model that only looks at the current token cannot do better than chance, because the current token is drawn independently. That control is what makes the attention number meaningful.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'torch', 'numpy'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import torch
import torch.nn as nn
import torch.nn.functional as F

from labs._shared.seed import seed_everything

seed_everything(0)
torch.manual_seed(0)
V, T = 10, 6


## Guided exercise: a copy-the-first-token attention model

The boolean attention mask uses `True` for positions that must not be attended, which is the PyTorch convention for `nn.MultiheadAttention`. The loss is next-token-free: every position is trained to predict token 0. Position 0 predicting itself is trivial. The accuracy that matters is on the later positions.


In [ ]:
class CopyAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, 32)
        self.pos = nn.Embedding(T, 32)
        self.attn = nn.MultiheadAttention(32, num_heads=1, batch_first=True)
        self.head = nn.Linear(32, V)

    def forward(self, idx):
        b, t = idx.shape
        x = self.emb(idx) + self.pos(torch.arange(t, device=idx.device))
        banned = torch.triu(torch.ones(t, t, dtype=torch.bool, device=idx.device), diagonal=1)
        hidden, weights = self.attn(x, x, x, attn_mask=banned, need_weights=True)
        return self.head(hidden), weights


class CurrentTokenProbe(nn.Module):
    """Control: the label is not a function of the current token."""

    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, 32)
        self.head = nn.Linear(32, V)

    def forward(self, idx):
        return self.head(self.emb(idx))


def batch(n, generator):
    return torch.randint(0, V, (n, T), generator=generator)


def train(model, steps, current_only=False):
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    g = torch.Generator().manual_seed(0)
    last = None
    for _ in range(steps):
        idx = batch(128, g)
        target = idx[:, :1].expand(idx.size(0), T)
        if current_only:
            logits = model(idx)
        else:
            logits, _ = model(idx)
        loss = F.cross_entropy(logits.reshape(-1, V), target.reshape(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        last = float(loss.detach())
    return last


copier = CopyAttention()
control = CurrentTokenProbe()
print("copier loss", round(train(copier, 80), 3), "control loss", round(train(control, 80, current_only=True), 3))

@torch.no_grad()
def evaluate(model, current_only=False):
    idx = batch(400, torch.Generator().manual_seed(1))
    target = idx[:, :1].expand_as(idx)
    if current_only:
        logits = model(idx)
        weights = None
    else:
        logits, weights = model(idx)
    pred = logits.argmax(dim=-1)
    acc_later = float((pred[:, 1:] == target[:, 1:]).float().mean())
    return acc_later, weights

acc_copy, weights = evaluate(copier)
acc_control, _ = evaluate(control, current_only=True)
# weights: (batch, query, key), averaged over heads. Mass on key 0 from queries 1:.
mass = float(weights[:, 1:, 0].mean())
print(f"attention mass on position 0 {mass:.3f}")
print(f"accuracy later positions, copier {acc_copy:.3f}, current-token probe {acc_control:.3f}")
assert mass > 0.45
assert acc_copy > acc_control + 0.25
print("LAB OK")


## Your turn

Print the attention row at query position 5, averaged over the evaluation batch. Position 0 should be the largest entry.

## Solution


In [ ]:
row = weights[:, 5, :].mean(dim=0)
print("mean attention from position 5", row.round(decimals=3).tolist())
assert int(row.argmax()) == 0


## Beyond the course

TransformerLens names the attention pattern `blocks.0.attn.hook_pattern` and the residual stream `blocks.0.hook_resid_post`. The cell builds a one-layer `HookedTransformer` from a config, with no download, and reads those tensors. If the package is absent the cell skips. The scientific claim of the lab does not depend on the import. The import is the bridge to the library the papers' code is usually written in.


In [ ]:
try:
    from transformer_lens import HookedTransformer, HookedTransformerConfig
except ImportError:
    print("SKIP TransformerLens import. Measurements above used the same quantities.")
    print("hook names to look up: blocks.0.attn.hook_pattern, blocks.0.hook_resid_post")
else:
    cfg = HookedTransformerConfig(
        n_layers=1,
        n_heads=1,
        d_model=32,
        d_head=32,
        d_mlp=64,
        d_vocab=V,
        n_ctx=T,
        act_fn="relu",
        normalization_type=None,
        attn_only=False,
    )
    hooked = HookedTransformer(cfg)
    tokens = torch.randint(0, V, (4, T))
    logits, cache = hooked.run_with_cache(tokens)
    pattern = cache["blocks.0.attn.hook_pattern"]
    resid = cache["blocks.0.hook_resid_post"]
    print("logits", tuple(logits.shape), "pattern", tuple(pattern.shape), "resid", tuple(resid.shape))
    assert pattern.shape[0] == 4
    assert resid.shape[-1] == 32


## Research questions

1. The attention mass on position 0 is high. What extra experiment would you need before saying the head's output is a copy of that token, rather than a mix that the linear head untangles?
2. The current-token probe is near chance. Alain and Bengio's probe can be high in a layer the model does not use. How is that different from this control?
3. Anthropic's induction-head papers study a two-layer circuit that copies a previous occurrence of the current token. What would you add to this one-layer task to make that circuit necessary?

## Further reading

- Alain and Bengio, *Understanding Intermediate Layers Using Linear Classifier Probes* (2016).
- Elhage et al., *A Mathematical Framework for Transformer Circuits* (Anthropic, 2021).
- Elhage et al., *Toy Models of Superposition* (Anthropic, 2022).
- Olsson et al., *In-context Learning and Induction Heads* (Anthropic, 2022).
- Bricken et al., *Towards Monosemanticity* (Anthropic, 2023).
- Nanda and Bloom, TransformerLens (MIT). https://github.com/TransformerLensOrg/TransformerLens
- Conmy et al., *Towards Automated Circuit Discovery* (DeepMind, 2023).
- The ARENA mechanistic-interpretability curriculum covers the same measurements at greater length. Those notebooks are not copied. https://www.arena.education/
